# SciRet · scale_15K · Full Pipeline (Kaggle T4)
**Run all cells top-to-bottom. Everything is self-contained.**

Prerequisites:
- Add `OPENAI_API_KEY` to Kaggle Secrets (Notebook → Settings → Secrets)
- Attach the CORD-19 dataset (`cord-19-research-challenge`) as a data source
- Attach your `sciret-queries` dataset containing `queries.json`

In [ ]:
# Install/upgrade required packages (Kaggle T4)
import subprocess, sys
pkgs = [
    'ragas==0.2.6',
    'openai',
    'langchain-openai',
    'langchain-huggingface',
    'sentence-transformers',
    'rank-bm25',
    'FlagEmbedding',
    'python-dotenv',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('Packages ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════
# SCALE CONFIG — only N_PAPERS and SCALE_LABEL change per tier
# ══════════════════════════════════════════════════════════════
import os, sys, json, random, pickle, time
import numpy as np
import pandas as pd
import torch

SCALE_LABEL   = "scale_15K"
N_PAPERS      = 15_000         # papers sampled from CORD-19
RANDOM_SEED   = 42
CHUNK_SIZE    = 400              # tokens
CHUNK_OVERLAP = 50
RRF_K         = 60              # standard RRF param
TOP_K_STAGE1  = 50              # candidates sent to reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

RESULTS_DIR = f"/kaggle/working/{SCALE_LABEL}"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale: {SCALE_LABEL} | N={15000:,} | Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# Load OPENAI_API_KEY from Kaggle Secrets
try:
    from kaggle_secrets import UserSecretsClient
    OPENAI_API_KEY = UserSecretsClient().get_secret('OPENAI_API_KEY')
    print('OpenAI key loaded from Kaggle Secrets ✓')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
    print('OpenAI key loaded from .env ✓' if OPENAI_API_KEY else 'WARNING: no OpenAI key found')
assert OPENAI_API_KEY, 'Add OPENAI_API_KEY to Kaggle Secrets → Settings → Add a Secret'

GEN_MODEL = 'gpt-4o-mini'
print(f'Model: {GEN_MODEL}')

## Step 1 · Sample & Chunk Papers

In [ ]:
# ── Load CORD-19 metadata ─────────────────────────────────────────────────
# Kaggle dataset path: /kaggle/input/cord-19-research-challenge/
CORD_META = '/kaggle/input/cord-19-research-challenge/metadata.csv'
assert os.path.exists(CORD_META), f'Attach CORD-19 dataset. Expected: {CORD_META}'

meta = pd.read_csv(CORD_META, low_memory=False)
meta = meta.drop_duplicates(subset=['cord_uid'])  # CORD-19 has duplicate cord_uid rows
print(f'Full CORD-19 (deduplicated): {len(meta):,} papers')

# Keep rows with usable abstract
meta = meta.dropna(subset=['abstract'])
meta = meta[meta['abstract'].str.len() > 100].reset_index(drop=True)
print(f'After abstract filter: {len(meta):,} papers')

# Stratified sample by year
meta['year'] = pd.to_numeric(meta['publish_time'].str[:4], errors='coerce')
meta = meta.dropna(subset=['year'])
meta['year'] = meta['year'].astype(int)

year_counts = meta['year'].value_counts()
year_fracs  = (year_counts / len(meta))
samples = []
for yr, frac in year_fracs.items():
    n = max(1, round(frac * N_PAPERS))
    sub = meta[meta['year'] == yr]
    samples.append(sub.sample(min(n, len(sub)), random_state=RANDOM_SEED))
df_papers = pd.concat(samples).head(N_PAPERS).reset_index(drop=True)
print(f'Sampled: {len(df_papers):,} papers (stratified by year)')
df_papers.to_parquet(f'{RESULTS_DIR}/sampled_papers.parquet', index=False)

In [ ]:
# ── Chunk papers ──────────────────────────────────────────────────────────
import re

def simple_token_count(text):
    return len(text.split())

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

chunk_records = []
for _, row in df_papers.iterrows():
    text = str(row.get('title', '')) + ' ' + str(row.get('abstract', ''))
    for j, chunk in enumerate(chunk_text(text)):
        chunk_records.append({
            'chunk_id': f"{row['cord_uid']}_{j}",
            'cord_uid': row['cord_uid'],
            'text':     chunk,
        })

df_chunks = pd.DataFrame(chunk_records)
df_chunks.to_parquet(f'{RESULTS_DIR}/chunks.parquet', index=False)
print(f'Total chunks: {len(df_chunks):,}')

## Step 2 · Embeddings + BM25 Index

In [ ]:
# ── BGE-M3 dense embeddings ──────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3', device=DEVICE)
texts = df_chunks['text'].tolist()
chunk_ids = df_chunks['chunk_id'].tolist()

BATCH = 128
all_embs = []
for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    embs = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embs.append(embs)
    if (i // BATCH) % 20 == 0:
        print(f'  {i+len(batch)}/{len(texts)} chunks embedded')

embeddings = np.vstack(all_embs)
np.save(f'{RESULTS_DIR}/bge_m3_embeddings.npy', embeddings)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
# ── BM25 index ────────────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi

tokenized = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tokenized)

with open(f'{RESULTS_DIR}/bm25_index.pkl', 'wb') as f:
    pickle.dump(bm25, f)
print('BM25 index saved.')

id_to_text = dict(zip(chunk_ids, texts))
id_to_cord = dict(zip(chunk_ids, df_chunks['cord_uid'].tolist()))

## Step 3 · Retrieval Ablation (Dense / BM25 / Hybrid)

In [ ]:
# ── Load evaluation queries ───────────────────────────────────────────────
# Attach your sciret-queries Kaggle dataset containing queries.json
QUERIES_PATH = '/kaggle/input/sciret-queries/queries.json'
if not os.path.exists(QUERIES_PATH):
    # Fallback: look in working dir
    QUERIES_PATH = f'{RESULTS_DIR}/queries.json'

with open(QUERIES_PATH) as f:
    QUERIES = json.load(f)
print(f'Loaded {len(QUERIES)} evaluation queries')

In [ ]:
# ── Retrieval helper functions ────────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

def dense_retrieve(query, k=TOP_K_STAGE1):
    q_emb = model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [(chunk_ids[i], float(sims[i])) for i in top_idx]

def bm25_retrieve(query, k=TOP_K_STAGE1):
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return [(chunk_ids[i], float(scores[i])) for i in top_idx]

def rrf_fuse(list_of_ranked, k=RRF_K):
    scores = {}
    for ranked in list_of_ranked:
        for rank, (cid, _) in enumerate(ranked):
            scores[cid] = scores.get(cid, 0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

def recall_at_k(ranked_ids, relevant_ids, k):
    top_k_cords = {id_to_cord[c] for c in ranked_ids[:k] if c in id_to_cord}
    rel = set(relevant_ids)
    return len(top_k_cords & rel) / len(rel) if rel else 0

print('Retrieval helpers defined.')

In [ ]:
# ── Recall@K evaluation (pseudo-ground-truth from hybrid 50K) ───────────
# NOTE: proper ground truth is in 1_data/eval/ground_truth.json
# On Kaggle, attach the sciret-queries dataset which also contains ground_truth.json
GT_PATH = '/kaggle/input/sciret-queries/ground_truth.json'
if os.path.exists(GT_PATH):
    with open(GT_PATH) as f:
        GROUND_TRUTH = json.load(f)
    print(f'Ground truth loaded: {len(GROUND_TRUTH)} queries')
else:
    print('WARNING: ground_truth.json not found — Recall@K will be skipped')
    GROUND_TRUTH = {}

results = {'dense': {}, 'bm25': {}, 'hybrid': {}}
for q in QUERIES:
    dense_res  = dense_retrieve(q)
    bm25_res   = bm25_retrieve(q)
    hybrid_res = rrf_fuse([dense_res, bm25_res])
    results['dense'][q]  = [c for c,_ in dense_res]
    results['bm25'][q]   = [c for c,_ in bm25_res]
    results['hybrid'][q] = [c for c,_ in hybrid_res]

if GROUND_TRUTH:
    recall_table = {}
    for sys_name, ranked_dict in results.items():
        row = {}
        for k in EVAL_K_VALUES:
            scores = []
            for q, ranked_ids in ranked_dict.items():
                if q in GROUND_TRUTH:
                    scores.append(recall_at_k(ranked_ids, GROUND_TRUTH[q], k))
            row[f'R@{k}'] = round(sum(scores)/len(scores), 4) if scores else float('nan')
        recall_table[sys_name] = row
    df_recall = pd.DataFrame(recall_table).T
    print('\n=== RECALL@K ===')
    print(df_recall.to_string())
    df_recall.to_csv(f'{RESULTS_DIR}/recall_at_k.csv')
    print(f'Saved: {RESULTS_DIR}/recall_at_k.csv')

## Step 4 · Cross-Encoder Reranking

In [ ]:
# ── Cross-encoder reranking ───────────────────────────────────────────────
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE)

reranked_results = {}
for q in QUERIES:
    candidates = results['hybrid'][q][:TOP_K_STAGE1]
    pairs = [(q, id_to_text[c]) for c in candidates if c in id_to_text]
    if not pairs:
        reranked_results[q] = candidates
        continue
    scores = reranker.predict(pairs)
    sorted_idx = np.argsort(scores)[::-1]
    reranked_results[q] = [candidates[i] for i in sorted_idx]

if GROUND_TRUTH:
    rerank_scores = {}
    for k in EVAL_K_VALUES:
        vals = [recall_at_k(reranked_results[q], GROUND_TRUTH.get(q, []), k)
                for q in QUERIES if q in GROUND_TRUTH]
        rerank_scores[f'R@{k}'] = round(sum(vals)/len(vals), 4) if vals else float('nan')
    df_rerank = pd.DataFrame([rerank_scores], index=['hybrid_reranked'])
    print('\n=== RERANKED RECALL@K ===')
    print(df_rerank.to_string())
    df_rerank.to_csv(f'{RESULTS_DIR}/reranked_recall_at_k.csv')
    print(f'Saved: {RESULTS_DIR}/reranked_recall_at_k.csv')

## Step 5 · Answer Generation + RAGAS Evaluation

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# Quick API test
resp = client.chat.completions.create(
    model=GEN_MODEL,
    messages=[{'role': 'user', 'content': 'Say hello in one word.'}],
    max_tokens=10,
)
print('API test OK:', resp.choices[0].message.content)

def generate_answer(query, context_chunks):
    ctx = '\n\n'.join([f'[{i+1}] {c}' for i, c in enumerate(context_chunks)])
    prompt = (
        'You are a scientific literature assistant. '
        'Answer the question using ONLY the provided context. '
        'Cite source numbers [1], [2], etc. for each claim.\n\n'
        f'Context:\n{ctx}\n\nQuestion: {query}\n\nAnswer:'
    )
    try:
        r = client.chat.completions.create(
            model=GEN_MODEL,
            messages=[{'role': 'user', 'content': prompt}],
            max_tokens=512, temperature=0.1,
        )
        return r.choices[0].message.content
    except Exception as e:
        return f'ERROR: {e}'

print('Generating answers...')
eval_records = []
for i, query in enumerate(QUERIES):
    top5_ids = reranked_results.get(query, results['hybrid'].get(query, []))[:5]
    contexts = [id_to_text[c] for c in top5_ids if c in id_to_text]
    answer   = generate_answer(query, contexts)
    eval_records.append({'question': query, 'answer': answer,
                         'contexts': contexts, 'ground_truth': query})
    if (i+1) % 10 == 0:
        print(f'  {i+1}/{len(QUERIES)} done')

df_eval = pd.DataFrame(eval_records)
df_eval.to_parquet(f'{RESULTS_DIR}/eval_runs.parquet', index=False)
errors = sum(1 for r in eval_records if str(r['answer']).startswith('ERROR:'))
print(f'Saved {len(eval_records)} answers. Errors: {errors}')

In [ ]:
# ── RAGAS Evaluation (OpenAI judge + local embeddings) ───────────────────
import ast
import numpy as np
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings

error_mask = df_eval['answer'].str.startswith('ERROR:')
if error_mask.any():
    print(f'Dropping {error_mask.sum()} error rows')
    df_eval = df_eval[~error_mask].reset_index(drop=True)

def to_list(val):
    if isinstance(val, list): return [str(v) for v in val]
    if isinstance(val, np.ndarray): return val.tolist()
    if isinstance(val, str):
        try: return ast.literal_eval(val)
        except: return [val]
    return list(val)

contexts_fixed = [to_list(c) for c in df_eval['contexts'].tolist()]

ragas_llm = LangchainLLMWrapper(
    ChatOpenAI(model=GEN_MODEL, api_key=OPENAI_API_KEY, temperature=0, max_tokens=512)
)
ragas_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
)
run_config = RunConfig(timeout=120, max_retries=5, max_workers=2)

ragas_ds = Dataset.from_dict({
    'question':     df_eval['question'].tolist(),
    'answer':       df_eval['answer'].tolist(),
    'contexts':     contexts_fixed,
    'ground_truth': df_eval['ground_truth'].tolist(),
})

print(f'Running RAGAS on {len(ragas_ds)} samples...')
result = evaluate(
    ragas_ds,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=ragas_llm, embeddings=ragas_emb, run_config=run_config,
)

In [ ]:
print("\n=== RAGAS RESULTS ===")
print(result) # Display the summary

# 1. Save detailed scores (individual rows) to CSV
out_csv = os.path.join(RESULTS_DIR, "ragas_scores.csv")
df_results = result.to_pandas()
df_results.to_csv(out_csv, index=False)
print(f"\nSaved per-query scores : {out_csv}")

# 2. Extract aggregate scores (averages) for the JSON file
# We calculate the mean directly from the numeric columns in the dataframe
agg_scores = df_results.select_dtypes(include=['number']).mean().to_dict()

# 3. Save aggregate scores to JSON
agg_path = os.path.join(RESULTS_DIR, "ragas_agg.json")
with open(agg_path, "w") as f:
    import json
    json.dump({
        "scale": SCALE_LABEL, 
        "judge_model": JUDGE_MODEL, 
        **agg_scores
    }, f, indent=2)
print(f"Saved aggregate scores  : {agg_path}")


In [ ]:
# ── Results ──────────────────────────────────────────────────────────────
import json as _json
print('\n=== RAGAS RESULTS ===')
scores = {}
for k, v in dict(result).items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
        scores[k] = v
    else:
        print(f'  {k}: {v}')

out_csv = f'{RESULTS_DIR}/ragas_scores.csv'
result.to_pandas().to_csv(out_csv, index=False)
print(f'\nSaved per-query scores : {out_csv}')

agg_path = f'{RESULTS_DIR}/ragas_agg.json'
with open(agg_path, 'w') as _f:
    _json.dump({'scale': SCALE_LABEL, 'n_papers': N_PAPERS,
                'n_ragas_samples': len(ragas_ds),
                'judge_model': GEN_MODEL, **scores}, _f, indent=2)
print(f'Saved aggregate scores  : {agg_path}')

In [ ]:
# ── Print final summary of all outputs ────────────────────────────────────
print(f'\n=== {SCALE_LABEL} COMPLETE ===')
for fname in sorted(os.listdir(RESULTS_DIR)):
    fpath = os.path.join(RESULTS_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<40} {size_kb:8.1f} KB')
print('\nDownload all files from the Output tab.')